# lasso_iv

> 对应代码：`EconometricsCode/code_in_notes/Chap.17/lasso_iv.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.17`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.17")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们清空全部内存并把最大变量数上限提高到 100000——后面 Lasso 方法要在大量候选工具变量中做选择，需要预留足够的变量空间。随后加载中国家庭金融调查（CHFS）的个体层面数据 chfs_ind.dta。

In [ ]:
%%stata
clear all
set maxvar 100000
use ../datasets/chfs_ind.dta,clear

我们构造回归所需的变量：exper 为工作经验（取自问卷变量 a3007）及其平方项 exper2，用以捕捉收入随经验的非线性（倒 U 型）关系；edu 为受教育年限（a2012），是我们关心的内生解释变量——教育可能同时与能力等不可观测因素相关，从而产生内生性。

In [ ]:
%%stata
gen exper=a3007
gen exper2=exper^2
gen edu=a2012

被解释变量取劳动收入的对数（加 1 以避免零收入取对数缺失），这是劳动经济学中明瑟方程（Mincer equation）的标准设定。

In [ ]:
%%stata
gen ln_income=log(1+labor_inc)

我们构造工具变量所需的出生队列信息：quarter 为出生季度（把月份 a2006 除以 4 后向下取整得到），birth_year 为出生年份（a2005）。这对应 Angrist 和 Krueger（1991）的经典做法：义务教育法使出生季度不同的人接受教育的时长系统不同，而出生季度本身不直接影响收入。

In [ ]:
%%stata
gen quarter=floor(a2006/4)
gen birth_year=a2005

最后我们用 poivregress（partialing-out IV Lasso 回归）估计教育的回报。括号内 (edu=i.quarter#i.province#i.birth_year) 声明内生变量 edu 以及候选工具变量集合——出生季度、省份、出生年份三个维度的完整交互，构成一个高维工具变量集；Lasso 从中自动遴选出预测力最强的工具变量组合。controls(exper*) 指定控制变量为经验及其平方；sel(cv) 表示用交叉验证（cross-validation）选择 Lasso 的惩罚参数 λ；cluster(province) 在省份层面聚类标准误。这种"后 Lasso"（post-Lasso）IV 方法在高维情形下既能提高工具变量的有效性，又能通过正交化矩条件保证推断的有效性。

In [ ]:
%%stata
poivregress ln_income (edu=i.quarter#i.province#i.birth_year), controls(exper*) sel(cv) cluster(province)